# Neural Network Regression with TensorFlow/Keras

This notebook covers:
- Building neural networks with TensorFlow and Keras
- Training regression models (predicting continuous values)
- Evaluating model performance with MAE and MSE
- Improving models by adding layers and training longer
- Saving and loading models
- Working with real-world data (insurance dataset)
- Data preprocessing: normalization and one-hot encoding

## Setup and Imports

In [ ]:
# Import TensorFlow - the main deep learning framework
import tensorflow as tf

# Check TensorFlow version (should be 2.x+)
# TensorFlow 2.x has Keras integrated as tf.keras
print(tf.__version__)

# Print current timestamp for tracking notebook runs
import datetime
print(f"Notebook last run (end-to-end): {datetime.datetime.now()}")

In [ ]:
# Import supporting libraries
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For plotting
import pandas as pd  # For data manipulation

## Part 1: Simple Regression Example

Let's start with a simple linear relationship: y = X + 10

In [ ]:
# Create features (input data) as TensorFlow constants
# These are the X values we'll use to predict y
X = tf.constant([-7.0, -4.0, -1.0, 2.0, 5.0, 8.0, 11.0, 14.0])

# Create labels (target data)
# Notice: y = X + 10 for all values
# The model's job is to learn this relationship
y = tf.constant([3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0])

# Visualize the linear relationship
plt.scatter(X, y);

In [ ]:
# Set random seed for reproducibility
# This ensures we get the same results each time we run the code
tf.random.set_seed(42)

# Build a simple neural network with Keras Sequential API
# Sequential: layers are stacked linearly (one after another)
model = tf.keras.Sequential([
    # Dense layer: fully connected layer with 1 neuron
    # This single neuron learns: y = weight * X + bias
    tf.keras.layers.Dense(1)
])

# Compile the model (configure for training)
model.compile(
    # Loss function: Mean Absolute Error
    # Measures average absolute difference between predictions and actual values
    loss=tf.keras.losses.mae,
    
    # Optimizer: Stochastic Gradient Descent
    # Updates model weights to minimize loss
    optimizer=tf.keras.optimizers.SGD(),
    
    # Metrics to track during training
    metrics=["mae"]
)

# Train the model
# expand_dims adds a dimension: shape (8,) -> (8, 1)
# This matches the expected input format for Dense layer
# epochs=100: pass through the entire dataset 100 times
model.fit(tf.expand_dims(X, axis=-1), y, epochs=100)

In [ ]:
# Inspect the input data
X, y

In [ ]:
# Make a prediction on a new value
# Input: 17.0, Expected output: 27.0 (since y = X + 10)
# Using np.array([[17.0]]) creates shape (1, 1) as expected by model
model.predict(np.array([[17.0]]))

## Part 2: Larger Dataset with Train/Test Split

In [ ]:
# Create a larger dataset
# X: values from -100 to 96 (step of 4)
X = np.arange(-100, 100, 4)

# y: same linear relationship (y = X + 10)
y = X + 10

In [ ]:
# Split data into training and testing sets
# 80% for training, 20% for testing
split_ind = int(0.8 * len(X))

# Training set: first 80% of data
# Used to train the model (adjust weights)
X_train = X[:split_ind] 
y_train = y[:split_ind]

# Test set: last 20% of data
# Used to evaluate model performance on unseen data
X_test = X[split_ind:] 
y_test = y[split_ind:]

len(X_train), len(X_test)

In [ ]:
# Visualize train and test split
plt.figure(figsize=(10, 7))

# Plot training data in blue
plt.scatter(X_train, y_train, c='b', label='Training data')

# Plot test data in green
# Test data is separate to evaluate generalization
plt.scatter(X_test, y_test, c='g', label='Testing data')

plt.legend();

In [ ]:
# Build a new model for the larger dataset
tf.random.set_seed(42)

model = tf.keras.Sequential([
    # Dense layer with 1 neuron
    # input_shape=[1] explicitly specifies we expect 1 feature
    tf.keras.layers.Dense(1, input_shape=[1])
])

# Compile the model
model.compile(
    loss=tf.keras.losses.mae,  # Mean Absolute Error
    optimizer=tf.keras.optimizers.SGD(),  # Stochastic Gradient Descent
    metrics=["mae"]
)

In [ ]:
# Display model architecture before training
# Shows: layers, output shapes, and number of parameters
model.summary()

In [ ]:
# Train the model on training data
# verbose=0: suppress training progress output for cleaner notebook
model.fit(X_train, y_train, epochs=100, verbose=0)

In [ ]:
# Display model summary after training
# Parameters are now trained (optimized)
model.summary()

## Making Predictions and Visualization

In [ ]:
# Make predictions on the test set
# Model has never seen this data during training
y_preds = model.predict(X_test)

In [ ]:
# Helper function to visualize predictions
def plot_predictions(train_data=X_train, 
                     train_labels=y_train, 
                     test_data=X_test, 
                     test_labels=y_test, 
                     predictions=y_preds):
    """
    Plots training data, test data, and predictions.
    
    Args:
        train_data: Training features
        train_labels: Training targets
        test_data: Test features
        test_labels: Test targets
        predictions: Model predictions on test data
    """
    plt.figure(figsize=(10, 7))
    
    # Plot training data in blue
    plt.scatter(train_data, train_labels, c="b", label="Training data")
    
    # Plot test data in green
    plt.scatter(test_data, test_labels, c="g", label="Testing data")
    
    # Plot predictions in red
    # If predictions overlap with test data, the model is accurate
    plt.scatter(test_data, predictions, c="r", label="Predictions")
    
    plt.legend();

In [ ]:
# Visualize how well predictions match test data
# Red dots (predictions) should overlap with green dots (actual test data)
plot_predictions(
    train_data=X_train,
    train_labels=y_train,
    test_data=X_test,
    test_labels=y_test,
    predictions=y_preds
)

In [ ]:
# Evaluate model on test set
# Returns [loss, mae] - both should be low for good performance
model.evaluate(X_test, y_test)

In [ ]:
# Manually calculate Mean Absolute Error
# squeeze() removes extra dimensions from predictions
mae = tf.metrics.mae(y_true=y_test, y_pred=y_preds.squeeze())
mae

## Part 3: Experimenting with Different Architectures

Let's compare models with different configurations:
- Model 1: 1 layer, 100 epochs
- Model 2: 2 layers, 100 epochs
- Model 3: 2 layers, 500 epochs

In [ ]:
### Model 1: 1 layer, 100 epochs (baseline)
tf.random.set_seed(42)

model_1 = tf.keras.Sequential([
    tf.keras.layers.Dense(1)  # Single output neuron
])

model_1.compile(
    loss=tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["mae"]
)

# Train for 100 epochs
model_1.fit(tf.expand_dims(X_train, axis=-1), y_train, epochs=100, verbose=0)

In [ ]:
### Model 2: 2 layers, 100 epochs
# Adding a second layer to see if it improves performance
tf.random.set_seed(42)

model_2 = tf.keras.Sequential([
    tf.keras.layers.Dense(1),
    tf.keras.layers.Dense(1)  # Additional layer
])

model_2.compile(
    loss=tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["mae"]
)

# Train for same number of epochs as model_1
model_2.fit(tf.expand_dims(X_train, axis=-1), y_train, epochs=100, verbose=0)

In [ ]:
### Model 3: 2 layers, 500 epochs
# Same architecture as model_2 but trained for longer
tf.random.set_seed(42)

model_3 = tf.keras.Sequential([
    tf.keras.layers.Dense(1),
    tf.keras.layers.Dense(1)
])

model_3.compile(
    loss=tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["mae"]
)

# Train for 500 epochs (5x more than models 1 and 2)
model_3.fit(tf.expand_dims(X_train, axis=-1), y_train, epochs=500, verbose=0)

## Comparing Model Performance

In [ ]:
# Define metric functions for comparison
def mae(y_test, y_pred):
    """
    Calculates Mean Absolute Error between true and predicted values.
    MAE = average of |predicted - actual|
    Lower is better.
    """
    return tf.metrics.mae(y_test, y_pred)
  
def mse(y_test, y_pred):
    """
    Calculates Mean Squared Error between true and predicted values.
    MSE = average of (predicted - actual)²
    Penalizes larger errors more than MAE.
    Lower is better.
    """
    return tf.metrics.mse(y_test, y_pred)

In [ ]:
# Make predictions with all three models
y_preds_1 = model_1.predict(X_test)
y_preds_2 = model_2.predict(X_test)
y_preds_3 = model_3.predict(X_test)

In [ ]:
# Calculate metrics for all models
# squeeze() removes extra dimensions from predictions
# .numpy() converts TensorFlow tensor to NumPy array

# Model 1 metrics
mae_1 = mae(y_test, y_preds_1.squeeze()).numpy()
mse_1 = mse(y_test, y_preds_1.squeeze()).numpy()

# Model 2 metrics
mae_2 = mae(y_test, y_preds_2.squeeze()).numpy()
mse_2 = mse(y_test, y_preds_2.squeeze()).numpy()

# Model 3 metrics
mae_3 = mae(y_test, y_preds_3.squeeze()).numpy()
mse_3 = mse(y_test, y_preds_3.squeeze()).numpy()

In [ ]:
# Create comparison table
model_results = [
    ["model_1", mae_1, mse_1],
    ["model_2", mae_2, mse_2],
    ["model_3", mae_3, mse_3]
]

In [ ]:
# Convert to pandas DataFrame for better visualization
all_results = pd.DataFrame(model_results, columns=["model", "mae", "mse"])

In [ ]:
# Display comparison table
# Lower MAE and MSE indicate better performance
# Model 3 should perform best (trained longest)
all_results

## Part 4: Saving and Loading Models

In [ ]:
# Save the best model (model_2 in this case)
# Two formats available:
# 1. SavedModel format (directory): model.save('model_name')
# 2. HDF5 format (single file): model.save('model_name.keras')

# We'll use HDF5 format (.keras extension)
model_2.save("best_model_HDF5_format.keras")

In [ ]:
# Load the saved model
# The loaded model should have identical architecture and weights
loaded_saved_model = tf.keras.models.load_model("best_model_HDF5_format.keras")

# Verify the architecture matches
loaded_saved_model.summary()

In [ ]:
# Verify loaded model produces identical predictions
model_2_preds = model_2.predict(X_test)
saved_model_preds = loaded_saved_model.predict(X_test)

# Compare MAE - should be exactly the same
mae(y_test, saved_model_preds.squeeze()).numpy() == mae(y_test, model_2_preds.squeeze()).numpy()

## Part 5: Real-World Dataset - Insurance Prediction

Predicting insurance charges based on:
- Age, BMI, number of children (numerical features)
- Sex, smoker status, region (categorical features)

In [ ]:
# Load insurance dataset from GitHub
insurance = pd.read_csv("https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv")

In [ ]:
# Explore the dataset
# Contains both numerical and categorical features
insurance.head()

In [ ]:
# Convert categorical features to numerical using one-hot encoding
# One-hot encoding: 'male' -> [1, 0], 'female' -> [0, 1]
# dtype=int ensures binary values (0 or 1) instead of boolean
insurance_one_hot = pd.get_dummies(insurance, dtype=int)

# View the transformed data
insurance_one_hot.head()

In [ ]:
# Separate features (X) and target (y)
# X: all columns except 'charges'
X = insurance_one_hot.drop("charges", axis=1)

# y: insurance charges (what we want to predict)
y = insurance_one_hot["charges"]

In [ ]:
# View features
X.head()

In [ ]:
# Import train_test_split from scikit-learn
from sklearn.model_selection import train_test_split

In [ ]:
# Split data into training and test sets
# test_size=0.15: 85% training, 15% testing
# random_state=42: ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

In [ ]:
# Build first insurance model (simple architecture)
tf.random.set_seed(42)

insurance_model = tf.keras.Sequential([
    tf.keras.layers.Dense(1),  # Hidden layer
    tf.keras.layers.Dense(1)   # Output layer
])

insurance_model.compile(
    loss=tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["mae"]
)

# Train for 100 epochs
insurance_model.fit(X_train, y_train, epochs=100, verbose=0)

In [ ]:
# Evaluate the first insurance model
# Performance likely won't be great due to simple architecture
insurance_model.evaluate(X_test, y_test)

## Improving the Insurance Model

In [ ]:
### Improved architecture with more neurons and better optimizer
tf.random.set_seed(42)

insurance_model_2 = tf.keras.Sequential([
    # Larger first layer to capture more patterns
    tf.keras.layers.Dense(100),
    # Intermediate layer
    tf.keras.layers.Dense(10),
    # Output layer (1 neuron for regression)
    tf.keras.layers.Dense(1)
])

insurance_model_2.compile(
    loss=tf.keras.losses.mae,
    # Adam optimizer: adaptive learning rate (usually better than SGD)
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["mae"]
)

# Train and save history for plotting
history = insurance_model_2.fit(X_train, y_train, epochs=100, verbose=0)

In [ ]:
# Evaluate improved model
# Should show significant improvement over insurance_model
insurance_model_2.evaluate(X_test, y_test)

In [ ]:
# Extract evaluation metrics
insurance_model_2_loss, insurance_model_2_mae = insurance_model_2.evaluate(X_test, y_test)

In [ ]:
# Plot training history (loss curve)
# Shows how loss decreases over epochs
df = pd.DataFrame(history.history)

df.plot()
plt.ylabel("loss")
plt.xlabel("epochs")
plt.show()

In [ ]:
# Train for more epochs to see if performance improves further
history_2 = insurance_model_2.fit(X_train, y_train, epochs=200, verbose=0)

In [ ]:
# Plot extended training history
# Check if loss continues to decrease or plateaus
df = pd.DataFrame(history_2.history)

df.plot()
plt.ylabel("loss")
plt.xlabel("epochs")
plt.show()

## Part 6: Preprocessing - Normalization and One-Hot Encoding

Proper preprocessing can significantly improve model performance.

In [ ]:
# Reload original dataset (before one-hot encoding)
insurance = pd.read_csv("https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv")

In [ ]:
# Import preprocessing tools from scikit-learn
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

In [ ]:
# Create column transformer for preprocessing
# Applies different transformations to different columns
ct = make_column_transformer(
    # MinMaxScaler: scales numerical features to [0, 1] range
    # This helps neural networks learn more efficiently
    (MinMaxScaler(), ["age", "bmi", "children"]),
    
    # OneHotEncoder: converts categorical features to binary columns
    # handle_unknown="ignore": handles new categories in test data
    (OneHotEncoder(handle_unknown="ignore"), ["sex", "smoker", "region"])
)

# Create features and target
X = insurance.drop("charges", axis=1)
y = insurance["charges"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit transformer on training data only
# IMPORTANT: Fitting on test data would cause data leakage!
ct.fit(X_train)

# Transform both training and test data
X_train_normal = ct.transform(X_train)
X_test_normal = ct.transform(X_test)

In [ ]:
# Compare original (non-normalized) data
# Values are on different scales: age (19), bmi (27.9), children (0)
X_train.loc[0]

In [ ]:
# Compare normalized and one-hot encoded data
# All values are now between 0 and 1
# Categorical features are binary (0 or 1)
X_train_normal[0]

In [ ]:
### Train model on preprocessed data
tf.random.set_seed(42)

# Same architecture as insurance_model_2
insurance_model_3 = tf.keras.Sequential([
    tf.keras.layers.Dense(100),
    tf.keras.layers.Dense(10),
    tf.keras.layers.Dense(1)
])

insurance_model_3.compile(
    loss=tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['mae']
)

# Train on normalized data
# Should perform better than model_2 due to preprocessing
insurance_model_3.fit(X_train_normal, y_train, epochs=200, verbose=0)

In [ ]:
# Evaluate model on normalized test data
insurance_model_3_loss, insurance_model_3_mae = insurance_model_3.evaluate(X_test_normal, y_test)

In [ ]:
# Compare model_2 (no preprocessing) vs model_3 (with preprocessing)
# Model 3 should have lower MAE due to normalized features
insurance_model_2_mae, insurance_model_3_mae